# Beyond Thought Anchors — Reproduce Whitebox Methods

**Method 2**: Receiver Head Attention Analysis  
**Method 3**: Causal Masking (Sentence-level KL Suppression)  

Model: `qwen-14b` = `deepseek-ai/DeepSeek-R1-Distill-Qwen-14B` (48 layers, 40 heads)  
Datasets: GPQA (20 questions) + Math (20 questions), correct base CoT only

**Output files:**
- Method 2 GPQA: `csvs/gpqa/receiver_head_scores_correct_qwen-14b_k32_pi16.csv`
- Method 2 Math: `csvs/math/receiver_head_scores_correct_qwen-14b_k32_pi16.csv`
- Method 3 GPQA: `kl_results/gpqa/qwen-14b/correct/problem_N_kl.npy`
- Method 3 Math: `kl_results/math/qwen-14b/correct/problem_N_kl.npy`
- Plots: `plots/`

## 0. Setup

In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.device_count())
print(torch.__version__)

!nvidia-smi

In [ ]:
import os

# 把 HuggingFace 模型缓存指向 50GB 数据盘，避免撑爆 30GB 系统盘
os.environ["HF_HOME"] = "/root/hf_cache"
os.environ["TRANSFORMERS_CACHE"] = "/root/hf_cache"

print(os.getcwd())

!pip install -r requirements.txt

---
## Method 2: Receiver Head Attention Analysis

**Step 1** `prep_attn_cache.py` — load 14B model, run 1 forward pass per problem, cache all 48x40 attention matrices to `attn_cache/`.

**Step 2** `generate_rec_csvs.py` — read .npy cache, compute receiver head scores, output CSV. No model loading needed.

> After Step 1 completes, Step 2 can run independently (cache persists on disk).

### Method 2 — GPQA

In [ ]:
# Step 1: Cache attention matrices for GPQA (1 forward pass per problem)
!python whitebox-analyses/scripts/prep_attn_cache.py --model qwen-14b --dataset gpqa --skip-receiver

In [ ]:
# Step 2: Generate receiver head CSV for GPQA
# Output: csvs/gpqa/receiver_head_scores_correct_qwen-14b_k32_pi16.csv
!python whitebox-analyses/scripts/generate_rec_csvs.py --model-name qwen-14b --data-model qwen-14b --dataset gpqa --top-k 32 --proximity-ignore 16 --output-dir csvs/gpqa

### Method 2 — Math

In [ ]:
# Step 1: Cache attention matrices for Math
!python whitebox-analyses/scripts/prep_attn_cache.py --model qwen-14b --dataset math --skip-receiver

In [ ]:
# Step 2: Generate receiver head CSV for Math
# Output: csvs/math/receiver_head_scores_correct_qwen-14b_k32_pi16.csv
!python whitebox-analyses/scripts/generate_rec_csvs.py --model-name qwen-14b --data-model qwen-14b --dataset math --top-k 32 --proximity-ignore 16 --output-dir csvs/math

---
## Method 3: Causal Masking (KL Suppression)

For each sentence in each problem, mask that sentence's attention and re-run forward pass. Compute token-level KL divergence to get a sentence×sentence causal influence matrix.

**N_sentences forward passes per problem** (~100 sentences/problem for GPQA).

**Output**: `kl_results/{dataset}/qwen-14b/correct/problem_N_kl.npy` (sentence×sentence matrix)

> Row sum → per-chunk causal importance.  
> `@pkld` caching: interrupted runs resume automatically, completed problems are skipped.

### Method 3 — GPQA

In [ ]:
# Output: kl_results/gpqa/qwen-14b/correct/problem_N_kl.npy
!python whitebox-analyses/scripts/prep_suppression_mtxs.py --model-name qwen-14b --dataset gpqa --output-dir kl_results/gpqa

### Method 3 — Math

In [ ]:
# Output: kl_results/math/qwen-14b/correct/problem_N_kl.npy
!python whitebox-analyses/scripts/prep_suppression_mtxs.py --model-name qwen-14b --dataset math --output-dir kl_results/math

---
## Verify Output Files

In [3]:
import os
import numpy as np
import pandas as pd

print("=== Method 2: Receiver Head CSVs ===")
for dataset in ["gpqa", "math"]:
    path = f"csvs/{dataset}/receiver_head_scores_correct_qwen-14b_k32_pi16.csv"
    if os.path.exists(path):
        df = pd.read_csv(path)
        n_problems = df["problem_number"].nunique()
        print(f"  [{dataset}] {path} - {n_problems} problems, {len(df)} sentences")
    else:
        print(f"  [{dataset}] MISSING: {path}")

print()
print("=== Method 3: KL Suppression Matrices ===")
for dataset in ["gpqa", "math"]:
    kl_dir = f"kl_results/{dataset}/qwen-14b/correct"
    if os.path.exists(kl_dir):
        files = [f for f in os.listdir(kl_dir) if f.endswith(".npy")]
        print(f"  [{dataset}] {kl_dir} - {len(files)} matrices")
        for f in sorted(files)[:3]:
            mat = np.load(os.path.join(kl_dir, f))
            print(f"    {f}: shape={mat.shape}")
    else:
        print(f"  [{dataset}] MISSING: {kl_dir}")

=== Method 2: Receiver Head CSVs ===
  [gpqa] csvs/gpqa/receiver_head_scores_correct_qwen-14b_k32_pi16.csv - 20 problems, 2721 sentences
  [math] csvs/math/receiver_head_scores_correct_qwen-14b_k32_pi16.csv - 20 problems, 2932 sentences

=== Method 3: KL Suppression Matrices ===
  [gpqa] kl_results/gpqa/qwen-14b/correct - 20 matrices
    problem_0_kl.npy: shape=(198, 198)
    problem_10_kl.npy: shape=(204, 204)
    problem_11_kl.npy: shape=(242, 242)
  [math] kl_results/math/qwen-14b/correct - 20 matrices
    problem_1591_kl.npy: shape=(159, 159)
    problem_2050_kl.npy: shape=(143, 143)
    problem_2137_kl.npy: shape=(161, 161)


### Script 1 — plot_kurt_stats.py
读取 `attn_cache/` 中所有 GPQA 问题的注意力矩阵（48层×40头），计算每个 head 的峰度（kurtosis），
输出散点图 + 直方图到 `plots/kurt_plots/`。

首次运行会遍历 20 问题×1920 npy 文件（纯 CPU I/O），约需数分钟，之后 pkld 缓存，秒级复现。

In [4]:
# 输出：plots/kurt_plots/kurt_layer_scatter_qwen-14b_pi16.png
#        plots/kurt_plots/kurt_hist_qwen-14b_pi16.png
!python whitebox-analyses/scripts/plot_kurt_stats.py --model-name qwen-14b

### Script 2 — plot_rec_taxonomy.py
从 CSV 读取 receiver head 分数，按句子类别（plan_generation / fact_retrieval 等）画箱线图。

分别对两个数据集各跑一次，输出到 `plots/`。

In [8]:
# GPQA：输出 plots/receiver_taxonomy_qwen-14b_k32_pre_conv_pi16.png
!python whitebox-analyses/scripts/plot_rec_taxonomy.py \
    --model-name qwen-14b \
    --csv-dir csvs/gpqa \
    --output-dir plots/rec_taxonomy_gpqa

# Math：输出 plots/rec_taxonomy_math/receiver_taxonomy_qwen-14b_k32_pre_conv_pi16.png
!python whitebox-analyses/scripts/plot_rec_taxonomy.py \
    --model-name qwen-14b \
    --csv-dir csvs/math \
    --output-dir plots/rec_taxonomy_math

Filtered to 1458 pre-convergence sentences

Pairwise comparisons:
  fact_retrieval vs active_computation: t=2.09, p=0.055, N=15
  fact_retrieval vs uncertainty_management: t=-2.20, p=0.047, N=14
  fact_retrieval vs result_consolidation: t=-0.93, p=0.403, N=5
  active_computation vs uncertainty_management: t=-3.89, p=0.002, N=14
  active_computation vs result_consolidation: t=-1.63, p=0.179, N=5
  uncertainty_management vs result_consolidation: t=0.04, p=0.967, N=4
/root/autodl-tmp/autoDL/whitebox-analyses/scripts/plot_rec_taxonomy.py:309: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  ax = sns.boxplot(

Plot saved to plots/rec_taxonomy_gpqa/receiver_taxonomy_qwen-14b_k32_pre_conv_pi16.png
Figure(700x250)
Filtered to 1412 pre-convergence sentences

Pairwise comparisons:
  plan_generation vs fact_retrieval: t=6.20, p=0.000, N=17
  plan_generation vs ac

### Script 3 — plot_attention_heads.py
用 `--top-k 32` 选出 kurtosis 最高的 32 个 receiver head，画它们在单个问题上的 vertical score 曲线。

`--problem-num 0`（GPQA 第 0 题，已缓存在 attn_cache）。

In [5]:
# 输出：plots/head_distributions/pn_0-True_head_None-0_k32_qwen-14b.png
!python whitebox-analyses/scripts/plot_attention_heads.py \
    --model-name qwen-14b \
    --top-k 32 \
    --problem-num 0

Too high: layer=np.int64(42), head=np.int64(33), np.nanmax(vert_scores)=np.float64(0.012863242998719215)
Figure(800x300)


### Script 4 — plot_suppression_matrix.py
读取预计算的 KL 矩阵（`kl_results/`），画句子×句子的 suppression 热图。

下面对 GPQA 和 Math 各画一个样例（纯 numpy，不加载模型）。

In [10]:
# GPQA — problem 0
!python whitebox-analyses/scripts/plot_suppression_matrix.py \
    --model-name qwen-14b \
    --kl-results-dir kl_results/gpqa \
    --problem-num 0 \
    --output-dir plots/suppression_matrix

# Math — problem 1591（第一个 math 问题）
!python whitebox-analyses/scripts/plot_suppression_matrix.py \
    --model-name qwen-14b \
    --kl-results-dir kl_results/math \
    --problem-num 1591 \
    --output-dir plots/suppression_matrix

Loading precomputed KL matrix from kl_results/gpqa/qwen-14b/correct/problem_0_kl.npy
Figure(600x600)
Loading precomputed KL matrix from kl_results/math/qwen-14b/correct/problem_1591_kl.npy
Figure(600x600)


### Script 5 — plot_suppression_line_unused.py
对每个问题找出 suppression 影响最大的 top-k 句子，分别画出它们对全文的 KL 曲线。

In [7]:
# GPQA — problem 0, 画 top-3 最重要句子的 suppression 折线
!python whitebox-analyses/scripts/plot_suppression_line_unused.py \
    --model-name qwen-14b \
    --kl-results-dir kl_results/gpqa \
    --problem-num 0 \
    --plot-top-k 3 \
    --output-dir plots/suppression_analysis

# Math — problem 1591
!python whitebox-analyses/scripts/plot_suppression_line_unused.py \
    --model-name qwen-14b \
    --kl-results-dir kl_results/math \
    --problem-num 1591 \
    --plot-top-k 3 \
    --output-dir plots/suppression_analysis

Loading precomputed KL matrix from kl_results/gpqa/qwen-14b/correct/problem_0_kl.npy
KL matrix shape: (198, 198)

Top 3 most impactful sentences:
  1. Sentence 0: mean KL = -6.891, affects 54 sentences
Saved plot to plots/suppression_analysis/qwen-14b/problem_0_True/suppress_sent_0.png
  2. Sentence 6: mean KL = -10.070, affects 24 sentences
Saved plot to plots/suppression_analysis/qwen-14b/problem_0_True/suppress_sent_6.png
  3. Sentence 8: mean KL = -10.502, affects 17 sentences
Saved plot to plots/suppression_analysis/qwen-14b/problem_0_True/suppress_sent_8.png
Loading precomputed KL matrix from kl_results/math/qwen-14b/correct/problem_1591_kl.npy
KL matrix shape: (159, 159)

Top 3 most impactful sentences:
  1. Sentence 0: mean KL = -12.643, affects 17 sentences
Saved plot to plots/suppression_analysis/qwen-14b/problem_1591_True/suppress_sent_0.png
  2. Sentence 1: mean KL = -14.166, affects 5 sentences
Saved plot to plots/suppression_analysis/qwen-14b/problem_1591_True/suppress_se

### Script 6 — eval_receiver_head_same_sentences.py
计算 top-k receiver head 之间 vertical score 的组内相关性（split-half reliability），打印统计值。

参数与 Script 1 一致，同样从 attn_cache 读取（首次运行较慢）。

In [9]:
!python whitebox-analyses/scripts/eval_receiver_head_same_sentences.py --model-name qwen-14b

Loading vert scores for all problems (reads attn_cache/, may take a few minutes)...
Loaded 20 problems.
Computing top-k receiver heads...
Top-32 heads: [[31, 34], [42, 33], [34, 19], [23, 5], [34, 17], [14, 35], [34, 15], [6, 16], [34, 16], [28, 32], [6, 19], [35, 15], [41, 26], [44, 33], [41, 2], [37, 17], [12, 4], [45, 9], [34, 39], [42, 20], [12, 2], [41, 18], [10, 28], [43, 21], [29, 36], [22, 36], [18, 11], [44, 31], [39, 39], [19, 12], [35, 35], [26, 27]]
Overall mean correlation: 0.493
All-head mean correlation: 0.281
